# First L0 processor example, version==0.9.0

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-607

See the associated:

  * Python module: [first_l0_processor.py](./first_l0_processor.py)
  * YAML file: [first_l0_processor.yaml](./first_l0_processor.yaml)

## 1. Initialization

In [ ]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
init_dask_cluster_eopf(scale=4)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

In [ ]:
# Other imports
import getpass
import os
import os.path as osp
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    os.environ.get("RSPY_HOST_USER", getpass.getuser()),
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

# For each data: 
# input_config_dir: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_dir'.
# output_data_dir: s3 bucket directory that will contain the generated data.
s1_short = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.short.yaml",
    "output_data_dir": f"{s3_output}/s1.short",
}
s1 = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.yaml",
    "output_data_dir": f"{s3_output}/s1",
}
s3 = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_dordop_payload.yaml",
    "output_data_dir": f"{s3_output}/s3",
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

In [ ]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./first_l0_processor.yaml"

In [ ]:
deploy_name = "first-l0-processor/sprint21-first-l0-processor"
await prefect_utils.wait_for_deployment(deploy_name)

## 3. Run Prefect flow for S1 short data (~1 minute)

In [ ]:
output_data_dir = s1_short["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1_short)

In [ ]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 4. Run Prefect flow for S1 full data (~30 minutes)

In [ ]:
output_data_dir = s1["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")
    
    local_report_dir = osp.join("./l0", "reports", "s1")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 5. Run Prefect flow for S3 full data (~20 minutes)

In [ ]:
output_data_dir = s3["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s3)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")

    local_report_dir = osp.join("./l0", "reports", "s3")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 6. Shutdown the dask clusters

In [ ]:
# You can scale the clusters to 0 workers
dask_gateway.scale_cluster(dask_cluster.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [ ]:
from importlib import reload
debug_flow = True

In [ ]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway, None)
    init_dask_cluster_eopf(scale=4)
    from resources.dask_utils import *
    dask_gateway = dask_gateway_eopf
    dask_client = dask_client_eopf
    dask_cluster = dask_cluster_eopf
    os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

In [14]:
if debug_flow:
    import first_l0_processor
    reload(first_l0_processor)
    results = first_l0_processor.first_l0_processor(**s1_short)
    display(results)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


14:43:08.971 | INFO    | prefect.engine - Created flow run 'original-myna' for flow 'first-l0-processor'

14:43:08.972 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/27da1dd3-b88d-418e-80a9-9ce51bdb03d4

14:43:09.002 | INFO    | prefect.task_runner.dask - Connecting to existing Dask cluster GatewayCluster<c0b570e41f14484aae7d6bd47d8295ca, status=running>

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


...

14:43:12.161 | INFO    | Task run 'dummy_catalog_save-0e9' - Start catalog saving

14:43:13.202 | INFO    | Task run 'dummy_catalog_save-0e9' - End (dummy) catalog saving:
{
  "type": "Catalog",
  "id": "stac-fastapi",
  "stac_version": "1.1.0",
  "description": "stac-fastapi",
  "links": [
    {
      "rel": "self",
      "href": "http://rs-server-catalog:8000/catalog/",
      "type": "application/json"
    },
    {
      "rel": "root",
      "href": "http://rs-server-catalog:8000/catalog/",
      "type": "application/json",
      "title": "RS-PYTHON STAC Catalog"
    },
    {
      "rel": "data",
      "href": "http://rs-server-catalog:8000/catalog/collections",
      "type": "application/json"
    },
    {
      "rel": "conformance",
      "href": "http://rs-server-catalog:8000/catalog/conformance",
      "type": "application/json",
      "title": "STAC/OGC conformance classes implemented by this server"
    },
    {
      "rel": "search",
      "href": "http://rs-server-catalog:8000/catalog/search",
      "type": "application/geo+json",
      "title": "STAC search",
      "method": "GET"
    },
    {
      "rel": "search",
      "href": "http://rs-server-catalog:8000/catalog/search",
      "type": "application/geo+json",
      "title": "STAC search",
      "method": "POST"
    },
    {
      "rel": "http://www.opengis.net/def/rel/ogc/1.0/queryables",
      "href": "http://rs-server-catalog:8000/catalog/queryables",
      "type": "application/schema+json",
      "title": "Queryables",
      "method": "GET"
    },
    {
      "rel": "service-desc",
      "href": "http://rs-server-catalog:8000/catalog/api",
      "type": "application/vnd.oai.openapi+json;version=3.0",
      "title": "OpenAPI service description"
    },
    {
      "rel": "service-doc",
      "href": "http://rs-server-catalog:8000/catalog/api.html",
      "type": "text/html",
      "title": "OpenAPI service documentation"
    }
  ],
  "conformsTo": [
    "http://www.opengis.net/spec/cql2/1.0/conf/basic-cql2",
    "http://www.opengis.net/spec/cql2/1.0/conf/cql2-json",
    "http://www.opengis.net/spec/cql2/1.0/conf/cql2-text",
    "http://www.opengis.net/spec/ogcapi-features-1/1.0/conf/core",
    "http://www.opengis.net/spec/ogcapi-features-1/1.0/conf/geojson",
    "http://www.opengis.net/spec/ogcapi-features-1/1.0/conf/oas30",
    "http://www.opengis.net/spec/ogcapi-features-3/1.0/conf/features-filter",
    "http://www.opengis.net/spec/ogcapi-features-3/1.0/conf/filter",
    "https://api.stacspec.org/v1.0.0-rc.2/item-search#filter",
    "https://api.stacspec.org/v1.0.0/collections",
    "https://api.stacspec.org/v1.0.0/collections/extensions/transaction",
    "https://api.stacspec.org/v1.0.0/core",
    "https://api.stacspec.org/v1.0.0/item-search",
    "https://api.stacspec.org/v1.0.0/item-search#fields",
    "https://api.stacspec.org/v1.0.0/item-search#query",
    "https://api.stacspec.org/v1.0.0/item-search#sort",
    "https://api.stacspec.org/v1.0.0/ogcapi-features",
    "https://api.stacspec.org/v1.0.0/ogcapi-features/extensions/transaction"
  ],
  "title": "RS-PYTHON STAC Catalog"
}

14:43:13.206 | INFO    | Task run 'dummy_catalog_save-0e9' - Finished in state Completed()

14:43:13.237 | INFO    | Flow run 'original-myna' - Finished in state Completed()

{'type': 'Catalog',
 'id': 'stac-fastapi',
 'stac_version': '1.1.0',
 'description': 'stac-fastapi',
 'links': [{'rel': 'self',
   'href': 'http://rs-server-catalog:8000/catalog/',
   'type': 'application/json'},
  {'rel': 'root',
   'href': 'http://rs-server-catalog:8000/catalog/',
   'type': 'application/json',
   'title': 'RS-PYTHON STAC Catalog'},
  {'rel': 'data',
   'href': 'http://rs-server-catalog:8000/catalog/collections',
   'type': 'application/json'},
  {'rel': 'conformance',
   'href': 'http://rs-server-catalog:8000/catalog/conformance',
   'type': 'application/json',
   'title': 'STAC/OGC conformance classes implemented by this server'},
  {'rel': 'search',
   'href': 'http://rs-server-catalog:8000/catalog/search',
   'type': 'application/geo+json',
   'title': 'STAC search',
   'method': 'GET'},
  {'rel': 'search',
   'href': 'http://rs-server-catalog:8000/catalog/search',
   'type': 'application/geo+json',
   'title': 'STAC search',
   'method': 'POST'},
  {'rel': 'http